# Encoding — UCI Drug Consumption Dataset

This notebook encodes the ordinal drug usage labels (CL0–CL6) into a binary target variable for cannabis risk prediction, and saves train/test splits ready for modelling.

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv('/Users/tg197682/Downloads/Computer Science Year 3/COMP3931_Individual_Project/student-addiction-risk-prediction/data/processed/uci_drug_clean.csv')
print('Shape:', df.shape)
df.head()

Shape: (1877, 30)


,Age,Gender,Education,Country,Ethnicity,Nscore,Escore,Oscore,Ascore,Cscore,...,Crack,Ecstasy,Heroin,Ketamine,Legalh,LSD,Meth,Mushrooms,Nicotine,VSA
0,0.49788,0.48246,-0.05921,0.96082,0.12600,0.31287,-0.57545,-0.58331,-0.91699,-0.00665,...,CL0,CL0,CL0,CL0,CL0,CL0,CL0,CL0,CL2,CL0
1,-0.07854,-0.48246,1.98437,0.96082,-0.31685,-0.67825,1.93886,1.43533,0.76096,-0.14277,...,CL0,CL4,CL0,CL2,CL0,CL2,CL3,CL0,CL4,CL0
2,0.49788,-0.48246,-0.05921,0.96082,-0.31685,-0.46725,0.80523,-0.84732,-1.62090,-1.01450,...,CL0,CL0,CL0,CL0,CL0,CL0,CL0,CL1,CL0,CL0
3,-0.95197,0.48246,1.16365,0.96082,-0.31685,-0.14882,-0.80615,-0.01928,0.59042,0.58489,...,CL0,CL0,CL0,CL2,CL0,CL0,CL0,CL0,CL2,CL0
4,0.49788,0.48246,1.98437,0.96082,-0.31685,0.73545,-1.63340,-0.45174,-0.30172,1.30612,...,CL0,CL1,CL0,CL0,CL1,CL0,CL0,CL2,CL2,CL0


In [ ]:
# The drug columns use ordinal class labels from the UCI study:
#   CL0 = Never used
#   CL1 = Over a decade ago
#   CL2 = Last decade
#   CL3 = Last year
#   CL4 = Last month
#   CL5 = Last week
#   CL6 = Last day

# focusing on Cannabis as the target drug.
# Rationale: cannabis is the most commonly used illicit drug among students,
# has the largest and most balanced class distribution in this dataset,
# and is directly relevant to student drug addiction risk prediction.

print('Cannabis class distribution:')
print(df['Cannabis'].value_counts().sort_index())

Cannabis class distribution:
Cannabis
CL0    413
CL1    207
CL2    266
CL3    210
CL4    138
CL5    185
CL6    458
Name: count, dtype: int64


In [5]:
# Binarise the Cannabis target:
#   Non-user (0) = CL0 (never used) + CL1 (used over a decade ago)
#   At Risk  (1) = CL2 through CL6 (used within the last decade or more recently)
#
# Justification: CL2 onwards indicates ongoing or recent drug involvement,
# which aligns with the project objective of identifying at-risk students.

df['cannabis_risk'] = df['Cannabis'].apply(lambda x: 0 if x in ['CL0', 'CL1'] else 1)

print('Binary cannabis_risk distribution:')
print(df['cannabis_risk'].value_counts())
print()
print(f"Non-user (0): {(df['cannabis_risk']==0).sum()} ({(df['cannabis_risk']==0).mean()*100:.1f}%)")
print(f"At Risk  (1): {(df['cannabis_risk']==1).sum()} ({(df['cannabis_risk']==1).mean()*100:.1f}%)")

Binary cannabis_risk distribution:
cannabis_risk
1    1257
0     620
Name: count, dtype: int64

Non-user (0): 620 (33.0%)
At Risk  (1): 1257 (67.0%)


- The binary encoding produces a moderately imbalanced dataset (~33% Non-user, ~67% At Risk).
- This imbalance will be handled during modelling using `class_weight='balanced'` and sample weighting.

In [6]:
# Define features and target
FEATURES = ['Age', 'Gender', 'Education', 'Country', 'Ethnicity',
            'Nscore', 'Escore', 'Oscore', 'Ascore', 'Cscore', 'Impulsive', 'SS']

X = df[FEATURES]
y = df['cannabis_risk']

print('Features:', FEATURES)
print('X shape:', X.shape)
print('y shape:', y.shape)

Features: ['Age', 'Gender', 'Education', 'Country', 'Ethnicity', 'Nscore', 'Escore', 'Oscore', 'Ascore', 'Cscore', 'Impulsive', 'SS']
X shape: (1877, 12)
y shape: (1877,)


In [7]:
# Drop the raw drug columns — not used as features, only cannabis_risk is the target
# Keep only features + target for the final encoded dataset

df_encoded = df[FEATURES + ['cannabis_risk']].copy()
print('Encoded dataset shape:', df_encoded.shape)
df_encoded.head()

Encoded dataset shape: (1877, 13)


,Age,Gender,Education,Country,Ethnicity,Nscore,Escore,Oscore,Ascore,Cscore,Impulsive,SS,cannabis_risk
0,0.49788,0.48246,-0.05921,0.96082,0.12600,0.31287,-0.57545,-0.58331,-0.91699,-0.00665,-0.21712,-1.18084,0
1,-0.07854,-0.48246,1.98437,0.96082,-0.31685,-0.67825,1.93886,1.43533,0.76096,-0.14277,-0.71126,-0.21575,1
2,0.49788,-0.48246,-0.05921,0.96082,-0.31685,-0.46725,0.80523,-0.84732,-1.62090,-1.01450,-1.37983,0.40148,1
3,-0.95197,0.48246,1.16365,0.96082,-0.31685,-0.14882,-0.80615,-0.01928,0.59042,0.58489,-1.37983,-1.18084,1
4,0.49788,0.48246,1.98437,0.96082,-0.31685,0.73545,-1.63340,-0.45174,-0.30172,1.30612,-0.21712,-0.21575,1


In [8]:
# Stratified 80/20 train/test split
# stratify=y ensures class proportions are preserved in both sets

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Train: {len(X_train)} samples')
print(f'Test : {len(X_test)} samples')
print()
print('Train class distribution:')
print(y_train.value_counts())
print()
print('Test class distribution:')
print(y_test.value_counts())

Train: 1501 samples
Test : 376 samples

Train class distribution:
cannabis_risk
1    1005
0     496
Name: count, dtype: int64

Test class distribution:
cannabis_risk
1    252
0    124
Name: count, dtype: int64


In [9]:
# Save train and test splits
train_df = X_train.copy()
train_df['cannabis_risk'] = y_train.values

test_df = X_test.copy()
test_df['cannabis_risk'] = y_test.values

train_df.to_csv('uci_cannabis_train.csv', index=False)
test_df.to_csv('uci_cannabis_test.csv', index=False)

print('Saved: uci_cannabis_train.csv')
print('Saved: uci_cannabis_test.csv')

Saved: uci_cannabis_train.csv
Saved: uci_cannabis_test.csv
